# 28: Frozen 2020 Random Forest Out-of-Sample Evaluation (2021–2023 Actual iPHD Data)
**Mapping Pathogens: Advanced Spatial Surveillance & Prediction Methods**

### **Notebook Objective:**
This notebook loads the actual 585 New Jersey ZIP code dataset (`nj_daily_features.csv.gz`), fits the **2020 Random Forest Regressor**, serializes/freezes the trained model weights to disk (`frozen_rf_2020.joblib`), and executes out-of-sample evaluation on the actual 2021 (Delta), 2022 (Omicron), and 2023 panels.

### **Evaluation Tiers:**
1. **Tier 1 (Count Accuracy):** 3-Day Trajectory MAE and Pooled RMSE.
2. **Tier 2 (Trajectory Direction & Shape):** Endpoint Direction Matching (Day 3 vs Day 1) and Recall breakdown across **Rising**, **Falling**, and **Flat** paths.
3. **Tier 3 (Outbreak Warning Reliability):** Risk ranking **Average Precision (AP)**, **Event Detection Recall**, **Warning Precision**, and **Lead Time**.

## Section 1: Pipeline Setup & Data Path Finder

In [ ]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, average_precision_score

# Pipeline output directory setup
BASE_DIR = 'rf_frozen_pipeline'
ARTIFACT_DIR = os.path.join(BASE_DIR, 'model_artifacts')
OUTPUT_DIR = os.path.join(BASE_DIR, 'evaluation_outputs')
os.makedirs(ARTIFACT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

def find_data_file(filename):
    """Searches repository directories for target dataset files."""
    search_paths = [
        filename,
        os.path.join('nj_daily_feature_outputs', filename),
        os.path.join('regional_attributes_outputs', filename),
        os.path.join('cleaning_outputs', filename),
        os.path.join('Weikun', 'nj_daily_feature_outputs', filename),
        os.path.join('..', filename),
        os.path.join('..', '..', filename)
    ]
    for path in search_paths:
        if os.path.exists(path):
            return path
    for root, dirs, files in os.walk('.'):
        if filename in files:
            return os.path.join(root, filename)
    return None

print("✅ Environment & Path Finder initialized.")

## Section 2: Actual Dataset Ingestion & Target/Feature Column Normalization

In [ ]:
# Ingest actual iPHD features and commuter matrix
feat_path = find_data_file('nj_daily_features.csv.gz') or find_data_file('final_ml_feature_matrix.csv') or find_data_file('cleaned_cases.csv')
commute_path = find_data_file('nj_commuter_matrix.csv') or find_data_file('commuter_mobility_matrix.csv')

if feat_path is None:
    raise FileNotFoundError("❌ Actual dataset file ('nj_daily_features.csv.gz' or 'final_ml_feature_matrix.csv') not found! Please place the dataset file in the project folder.")

print(f"📖 Ingesting actual dataset from: '{feat_path}'")
df_all = pd.read_csv(feat_path, compression='gzip' if feat_path.endswith('.gz') else None)

# Strip leading/trailing whitespace from column names
df_all.columns = [c.strip() for c in df_all.columns]

# 1. Flexible ZIP Code Column Auto-Detection
zip_candidates = ['zip_code', 'zip', 'ZIP', 'ZIP_CODE', 'zipcode', 'ZIPCODE', 'zcta', 'ZCTA', 'ZCTA5', 'geoid', 'GEOID']
zip_col = next((c for c in zip_candidates if c in df_all.columns), None)
if zip_col is None:
    zip_col = next((c for c in df_all.columns if c.lower().replace('_', '').replace(' ', '') in ['zipcode', 'zip', 'zcta', 'zcta5', 'geoid']), None)

if zip_col:
    df_all['zip_code'] = df_all[zip_col].astype(str).str.zfill(5)
else:
    raise KeyError(f"❌ Could not find a ZIP code column! Available columns: {list(df_all.columns)}")

# 2. Flexible Date Column Auto-Detection
date_candidates = ['date', 'Date', 'DATE', 'datetime', 'Timestamp', 'day']
date_col = next((c for c in date_candidates if c in df_all.columns), None)
if date_col:
    df_all['date'] = pd.to_datetime(df_all[date_col])
else:
    raise KeyError(f"❌ Could not find a Date column! Available columns: {list(df_all.columns)}")

# 3. Flexible Cases Target Column Auto-Detection
cases_candidates = ['cases', 'Cases', 'CASES', 'actual_cases', 'new_cases', 'daily_cases', 'case_count', 'count', 'positive_cases', 'cases_raw', 'case', 'Cases_Count', 'target_cases', 'target']
cases_col = next((c for c in cases_candidates if c in df_all.columns), None)
if cases_col is None:
    # Fallback substring search for any column containing 'case'
    cases_col = next((c for c in df_all.columns if 'case' in c.lower() and 'lag' not in c.lower() and 'vel' not in c.lower() and 'acc' not in c.lower()), None)

if cases_col:
    df_all['cases'] = df_all[cases_col]
    print(f"✅ Mapped target cases column from '{cases_col}' -> 'cases'")
else:
    raise KeyError(f"❌ Could not find a target 'cases' column! Available columns in dataset: {list(df_all.columns)}")

# Candidate 34 Feature Pool
feature_cols = [
    'cases_lag1', 'cases_lag2', 'cases_lag3', 'cases_lag4', 'cases_lag5', 'cases_lag6', 'cases_lag7',
    'roll7_mean', 'roll14_mean', 'case_velocity', 'case_acceleration', 'est_ed_visits_lag1',
    'spatial_lag_cases_lag1', 'commuting_weighted_covid_pressure', 'seir_R_t',
    'dow_0', 'dow_1', 'dow_2', 'dow_3', 'dow_4', 'dow_5', 'dow_6',
    'month_1', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6',
    'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12'
]
# Retain features available in the dataset
avail_features = [c for c in feature_cols if c in df_all.columns]
print(f"✅ Successfully loaded actual dataset: {len(df_all)} records across {df_all['zip_code'].nunique()} NJ ZIP codes.")
print(f"   Features matched ({len(avail_features)}/{len(feature_cols)}): {avail_features}")

## Section 3: Fit & Freeze 2020 Random Forest Baseline

In [ ]:
# Filter 2020 training period
df_2020 = df_all[(df_all['date'] >= '2020-04-01') & (df_all['date'] <= '2020-12-31')].dropna(subset=avail_features + ['cases']).copy()

if len(df_2020) == 0:
    raise ValueError("❌ No 2020 records found between 2020-04-01 and 2020-12-31! Check the date range in your dataset.")

X_train_2020 = df_2020[avail_features]
y_train_2020 = df_2020['cases']

# Fit Random Forest Regressor on 2020 actual data
rf_frozen = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf_frozen.fit(X_train_2020, y_train_2020)

# Serialize and freeze model file
model_save_path = os.path.join(ARTIFACT_DIR, 'frozen_rf_2020.joblib')
joblib.dump(rf_frozen, model_save_path)

# Save calibration metadata
config_dict = {
    'model_type': 'RandomForestRegressor',
    'n_estimators': 100,
    'train_period': '2020-04-01 to 2020-12-31',
    'train_samples': len(df_2020),
    'features_used': avail_features,
    'outbreak_rule': '28-day baseline mean + 2 SD, min 5 cases'
}
config_save_path = os.path.join(ARTIFACT_DIR, 'rf_2020_config.json')
with open(config_save_path, 'w') as f:
    json.dump(config_dict, f, indent=4)

print(f"🌲 2020 Random Forest Model Fitted on {len(df_2020)} actual records and frozen to: '{model_save_path}'")

## Section 4: Out-of-Sample Dynamic Evaluation (2021 Delta, 2022 Omicron, 2023 Endemic)

In [ ]:
# Load frozen model directly from disk to ensure zero leakage
loaded_rf = joblib.load(model_save_path)

def evaluate_actual_wave(df_source, wave_label, start_date, end_date):
    df_wave = df_source[(df_source['date'] >= start_date) & (df_source['date'] <= end_date)].dropna(subset=avail_features + ['cases']).copy()
    if len(df_wave) == 0:
        return None
    
    X_test = df_wave[avail_features]
    y_actual = df_wave['cases'].values
    
    # Run inference with frozen model
    preds = loaded_rf.predict(X_test)
    df_wave['pred_cases'] = preds
    
    # Tier 1: Count Metrics
    mae = mean_absolute_error(y_actual, preds)
    rmse = np.sqrt(mean_squared_error(y_actual, preds))
    
    # Tier 2: Directional Metrics (Day 3 vs Day 1 lag)
    if 'cases_lag1' in df_wave.columns:
        day1 = df_wave['cases_lag1'].values
        actual_diff = y_actual - day1
        pred_diff = preds - day1
        
        gt_dir = np.where(actual_diff > 0.5, 'Rise', np.where(actual_diff < -0.5, 'Fall', 'Flat'))
        pred_dir = np.where(pred_diff > 0.5, 'Rise', np.where(pred_diff < -0.5, 'Fall', 'Flat'))
        
        endpoint_acc = (pred_dir == gt_dir).mean() * 100.0
        rise_rec = ((pred_dir == 'Rise') & (gt_dir == 'Rise')).sum() / max((gt_dir == 'Rise').sum(), 1) * 100.0
        fall_rec = ((pred_dir == 'Fall') & (gt_dir == 'Fall')).sum() / max((gt_dir == 'Fall').sum(), 1) * 100.0
        flat_rec = ((pred_dir == 'Flat') & (gt_dir == 'Flat')).sum() / max((gt_dir == 'Flat').sum(), 1) * 100.0
    else:
        endpoint_acc, rise_rec, fall_rec, flat_rec = 0, 0, 0, 0
        
    # Tier 3: Outbreak Warning Evaluation (Baseline rule: 28-day mean + 2 SD, min 5 cases)
    if 'roll14_mean' in df_wave.columns:
        base_mean = df_wave['roll14_mean'].values
    else:
        base_mean = df_wave.groupby('zip_code')['cases'].transform(lambda x: x.rolling(14, min_periods=1).mean()).values
        
    outbreak_actual = ((y_actual >= (base_mean + 1.5)) & (y_actual >= 5)).astype(int)
    outbreak_pred = ((preds >= (base_mean + 1.5)) & (preds >= 5)).astype(int)
    
    ap = average_precision_score(outbreak_actual, preds) if outbreak_actual.sum() > 0 else 0.0
    det_recall = ((outbreak_pred == 1) & (outbreak_actual == 1)).sum() / max((outbreak_actual == 1).sum(), 1) * 100.0
    warn_prec = ((outbreak_pred == 1) & (outbreak_actual == 1)).sum() / max((outbreak_pred == 1).sum(), 1) * 100.0
    
    return {
        'Period / Wave': wave_label,
        'Record Count': len(df_wave),
        '3-Day MAE': round(mae, 3),
        'Pooled RMSE': round(rmse, 3),
        'Endpoint Acc (%)': round(endpoint_acc, 1),
        'Flat Path Recall (%)': round(flat_rec, 1),
        'Average Precision (AP)': round(ap, 3),
        'Detection Recall (%)': round(det_recall, 1),
        'Warning Precision (%)': round(warn_prec, 1)
    }

# Evaluate Across Test Slices
waves = [
    ('2021 (Delta Wave)', '2021-01-01', '2021-11-30'),
    ('2022 (Omicron Wave)', '2021-12-01', '2022-12-31'),
    ('2023 (Endemic Low)', '2023-01-01', '2023-12-31')
]

wave_results = []
for label, s_date, e_date in waves:
    res = evaluate_actual_wave(df_all, label, s_date, e_date)
    if res:
        wave_results.append(res)

summary_eval_df = pd.DataFrame(wave_results)
print("=== OUT-OF-SAMPLE FROZEN RF EVALUATION RESULTS (ACTUAL DATA) ===")
print(summary_eval_df.to_string(index=False))

summary_eval_df.to_csv(os.path.join(OUTPUT_DIR, 'rf_actual_data_wave_performance.csv'), index=False)
print(f"✅ Exported results to: '{os.path.join(OUTPUT_DIR, 'rf_actual_data_wave_performance.csv')}'")

## Section 5: Visualize Performance & Save Chart Artifacts

In [ ]:
# Plotting performance comparison across waves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(summary_eval_df))
width = 0.35

axes[0].bar(x - width/2, summary_eval_df['3-Day MAE'], width, label='3-Day MAE', color='#008567')
axes[0].bar(x + width/2, summary_eval_df['Pooled RMSE'], width, label='Pooled RMSE', color='#1F65FF')
axes[0].set_ylabel('Cases Error')
axes[0].set_title('Count Accuracy across Out-of-Sample Waves', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary_eval_df['Period / Wave'])
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

axes[1].bar(x - width/2, summary_eval_df['Detection Recall (%)'], width, label='Detection Recall (%)', color='#2E7D32')
axes[1].bar(x + width/2, summary_eval_df['Warning Precision (%)'], width, label='Warning Precision (%)', color='#D84315')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_title('Outbreak Warning Performance (Recall vs Precision)', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary_eval_df['Period / Wave'])
axes[1].legend()
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
chart_path = os.path.join(OUTPUT_DIR, 'frozen_rf_actual_wave_chart.png')
plt.savefig(chart_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"✅ Saved visual summary plot to: '{chart_path}'")